# 머신러닝 파일별 2 · 2-1 펭귄 Train/Test · 교차검증 · GridSearchCV

**한 줄 요약**

- 전체로 학습
- 채점한 100% 착시에서 출발해 8 : 2 분할 · 교차검증
- GridSearchCV
- test 1회까지, '진짜 실력'을 재는 순서를 원본 문제 Q1 ~ Q12로 직접 풀기

- **원본** · `6.머닝러신/02_모델선택_펭귄/2-1_실습문제_펭귄_TrainTest_CV_GridSearch.ipynb` (원본 파일은 그대로 둠)
- **종합 정리본** · `00_최소선/05_머신러닝` 머신러닝-1 (데이터 분할 · 교차검증) 2 ~ 5장

## 🎯 학습 목표

이 노트북을 끝내면 **자료를 보지 않고** 다음을 할 수 있어야 함

1. 같은 데이터로 학습 · 채점한 1.0000이 왜 착시인지 설명한다
2. `train_test_split`으로 8 : 2 분할하고 test를 봉인하는 규칙을 지킨다
3. K-Fold를 for문으로 직접 구현하고 `cross_val_score` 한 줄과 비교한다
4. `GridSearchCV`로 `max_depth`를 고르고, 후보 범위는 사람이 정한다는 점을 말한다
5. 모든 선택을 끝낸 뒤 test를 한 번만 열어 최종 점수를 보고한다

## 어디에 쓰나

*실제 상황 → 이 권에서 배우는 것*

- **'정확도 100%'가 나왔을 때** · 같은 데이터로 채점했는지부터 (1-1)
- **시험 점수가 seed마다 다를 때** · 한 번의 분할 대신 교차검증 (1-4 · 2장)
- **하이퍼파라미터를 고를 때** · train 안의 교차검증 · GridSearchCV (3장)
- **보고서에 최종 성능을 적을 때** · test는 마지막 1회 (3-3)

## 학습 지도

<img src="images/pfm2_map.png" alt="2-1 펭귄 학습 지도" width="763">

**예상 학습 시간** · 개념 40분 + 실습 50분 (전체 실행 20초 이내 · GPU 필요 없음)

## 목차

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">장</th><th style="text-align: left;">제목</th><th style="text-align: left;">이 장의 질문</th></tr>
</thead>
<tbody>
<tr><td>0</td><td>준비</td><td>펭귄 342마리 · 원본 문제</td></tr>
<tr><td>1</td><td>착시와 Train / Test 분할</td><td>전체로 채점한 점수는 왜 믿을 수 없고, 한 번 나눈 시험 점수는 얼마나 믿을 만한가?</td></tr>
<tr><td>2</td><td>모델 교체와 교차검증</td><td>모델 셋 중 무엇이 좋은지, test를 건드리지 않고 어떻게 공정하게 비교하는가?</td></tr>
<tr><td>3</td><td>GridSearchCV와 최종 평가</td><td><code>max_depth</code>를 어떻게 고르고, 고른 모델의 최종 점수는 언제 · 어떻게 재는가?</td></tr>
<tr><td>4</td><td>종합 실습</td><td>원본 '오늘 정리' 표를 채우고, 같은 흐름을 KNN의 k에 적용하면?</td></tr>
<tr><td>—</td><td>마무리 · 부록</td><td>오늘 배운 것 · 기억 확인 · 오답 노트 · 면접 · 요약표 · 복습일 · 원본 대응표 · 전체 코드</td></tr>
</tbody></table>

---
## 0. 준비

펭귄 데이터는 원본 폴더의 `penguins.csv`를 읽음 (원본은 seaborn 내장 데이터 · 같은 344행). **셀은 위에서부터 차례로 실행**.

*항목 → 설정*

- **라이브러리** · pandas · numpy · scikit-learn
- **seed** · 분할 `random_state=7` · 모델 · KFold `random_state=42` (원본 정답과 같음)
- **데이터** · `../02_모델선택_펭귄/penguins.csv` → 측정값 결측 제거 342마리 × 4특성 · 3품종
- **경고** · 로지스틱 반복 횟수 경고만 숨김 (원본과 같음)

**실행 흐름**

1. `warnings` · `numpy` · `pandas` 불러오기
2. 로지스틱 반복 경고(UserWarning 계열) 숨기기
3. 원본 폴더(`../02_모델선택_펭귄/`)의 CSV 읽기 · 측정값 결측 제거 · X · y 만들기

In [1]:
# 준비: 1-2회차와 같은 펭귄 342마리 (측정값 결측만 제거)
# 왜 · 앞 회차와 같은 데이터로 시작해 결과를 이어서 비교하기 위해
import warnings  # 경고 숨기기

import numpy as np   # 평균 · 표준편차
import pandas as pd  # 표

# 로지스틱이 반복 횟수 경고를 띄움 (스케일링 전이라 어쩔 수 없음 · 원본과 같은 설정을 유지)
from sklearn.exceptions import ConvergenceWarning  # 반복 횟수 경고의 종류
warnings.filterwarnings("ignore", category=ConvergenceWarning)  # 반복 횟수 경고만 숨김 · 다른 경고는 그대로 보임

# 왜 · 인터넷 없이도 원본과 같은 데이터를 쓰도록 원본 폴더의 CSV를 상대 경로로 읽기 위해
CSV = "../02_모델선택_펭귄/penguins.csv"                    # 원본 폴더의 데이터 (복사하지 않음)
peng_raw = pd.read_csv(CSV)  # 원본은 sns.load_dataset("penguins") · 같은 344행
meas = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]  # 측정값 4개
peng = peng_raw.dropna(subset=meas).reset_index(drop=True)  # 측정값 결측 2마리 제거
X = peng[meas].values                                       # 특성 배열 (342, 4)
y = peng["species"].to_numpy()  # 정답 배열 · pandas 3는 .values가 Arrow 배열이라 to_numpy()
print(f"펭귄 {len(peng)}마리, 특성 {X.shape[1]}개")         # 342마리, 4개
print(f"품종: {sorted(set(y))}")                            # 3품종

펭귄 342마리, 특성 4개
품종: ['Adelie', 'Chinstrap', 'Gentoo']


### 원본 문제 · 규칙 · 변수 이름

- 1-2회차에서 342마리로 94.44%, 트리는 100%까지 나옴 → 그건 **이미 본 데이터** 점수
- 오늘 확인할 흐름 · Train/Test Split → 교차검증 → GridSearchCV → 최종 평가
- 실습 규칙 · `[예측]`이 있으면 실행 전에 먼저 적기 · 숫자가 나오면 "왜 이렇게 나왔나" 한 줄 적기
- 변수 이름 규약 (뒷부분이 앞 변수를 그대로 씀 · 이름을 똑같이)

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">만드는 곳</th><th style="text-align: left;">변수 이름</th><th style="text-align: left;">담기는 것</th></tr>
</thead>
<tbody>
<tr><td>준비</td><td><code>meas</code> · <code>peng</code> · <code>X</code> · <code>y</code></td><td>측정값 열 이름 · 342마리 표 · 특성 · 정답</td></tr>
<tr><td>Q3</td><td><code>X_train, X_test, y_train, y_test</code></td><td>8 : 2 분리 결과</td></tr>
<tr><td>Q4</td><td><code>dt</code> · <code>acc_split</code></td><td>모델 · 시험 정확도</td></tr>
<tr><td>Q7</td><td><code>fold_scores</code></td><td>5-Fold 각각의 정확도 (원본 표는 Q8로 적었지만 Q7에서 만듦)</td></tr>
<tr><td>Q11 · Q12</td><td><code>grid</code> · <code>best_model</code> · <code>final_acc</code></td><td>GridSearchCV · 최적 모델 · 최종 정확도</td></tr>
</tbody></table>

**원본 힌트**

- Test 데이터는 학습 중에 절대 들여다보지 않기 · 미리 열어 보는 순간 그 시험지는 무효 (데이터 누수)
- GridSearchCV는 '설계자'가 아니라 '탐색기' · 내가 넣은 후보 안에서만 비교

---
## 1. 착시와 Train / Test 분할

> **이 장의 흐름**
>
> - 전체로 학습 · 채점(1.0000) → 8 : 2로 나눠 처음 보는 test로 채점
> - 그다음 seed만 바꿔 시험 점수가 얼마나 흔들리는지 확인

<img src="images/pfm2_illusion.png" alt="같은 데이터 채점 1.0000 vs 처음 보는 test" width="677">

**파이프라인 위치** · ② 분할 → ⑥ 평가

### 1-1. `A` 전체로 학습하고 같은 데이터로 채점 · `fit(X, y)` · `predict(X)` <sub>원본 Q1</sub>

> 🎯 **전체 데이터로 학습하고 같은 데이터로 채점하면 몇 %가 나오고, 왜 믿으면 안 되는가?**

💡 **쉽게 말하면**

- 교과서로 공부하고 **그 교과서 문제**로 시험 보기.
- 트리는 끝까지 갈라 외울 수 있어 100%가 나옴 → 새 펭귄 실력은 모름.

**실행 흐름**

1. 트리를 전체 X, y로 학습
2. 같은 X로 예측
3. 정확도 계산

**먼저 예측** · 몇 %가 나올까? — 답은 코드 실행 결과 아래에

In [2]:
from sklearn.metrics import accuracy_score       # 정확도
from sklearn.tree import DecisionTreeClassifier  # 결정 트리

# 왜 · 학습에 쓴 데이터로 채점하면 점수가 부풀려지는 문제를 먼저 보이기 위해
dt_cheat = DecisionTreeClassifier(random_state=42)    # 원본 정답과 같은 seed
dt_cheat.fit(X, y)                                    # 342마리 전부로 공부
acc_cheat = accuracy_score(y, dt_cheat.predict(X))    # 공부한 그 문제로 채점
print(f"전체로 학습 + 전체로 채점: {acc_cheat:.4f}")  # 착시 점수
print("트리 잎 개수:", dt_cheat.get_n_leaves())       # 끝까지 갈라 외운 정도

전체로 학습 + 전체로 채점: 1.0000
트리 잎 개수: 14


**결과** · 예측 정답은 100% — 같은 문제로 채점했기 때문

- `전체로 학습 + 전체로 채점: 1.0000` — 외운 문제로 시험 → 만점 (착시)
- `트리 잎 개수: 14` — 잎 14개로 전부 갈라 외움

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>전체로 학습 + 채점</td><td>외운 문제로 시험</td><td>쓰지 않기 (점수가 부풀려짐)</td></tr>
<tr><td>Train / Test 분할 (1-2)</td><td>처음 보는 문제로 시험</td><td>성능을 재는 최소 조건</td></tr>
<tr><td>교차검증 (2장)</td><td>여러 번 나눠 평균</td><td>모델 · 설정 비교</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 1-2회차의 94.44% · 100%도 이미 본 데이터 점수였음
> - train 점수가 높다고 좋은 모델이 아님 → 처음 보는 데이터 점수로 판단

> ✅ **한 줄 정리** · 같은 데이터로 학습 · 채점하면 외운 문제로 시험 본 셈이라 1.0000이 나와도 새 데이터 실력은 알 수 없다.

**딥러닝 코드에서는**

- 학습 손실이 0에 가까워도 val 손실이 오르면 과적합 — 같은 착시
- 그래서 epoch마다 train · val 점수를 따로 기록

### 1-2. `A` 8 : 2로 나누고 test 봉인 · `train_test_split` · `stratify=y` <sub>원본 Q2 · Q3</sub>

> 🎯 **진짜 실력을 재려면 데이터를 어떻게 나누고, test는 언제 쓰는가?**

💡 **쉽게 말하면**

- Train은 **교과서**, Test는 **봉인된 시험지**.
- 성능이 궁금해 미리 열어 보는 순간 그 시험지는 무효 (데이터 누수).

**실행 흐름**

1. test_size=0.2 · stratify=y · random_state=7로 분할
2. 마리 수 확인
3. 품종 비율 확인

**먼저 예측** · 342마리를 8 : 2로 나누면 Train · Test는 몇 마리? — 답은 코드 실행 결과 아래에

In [3]:
from sklearn.model_selection import train_test_split  # 학습 · 시험 분할

# 왜 · 평가용 데이터를 학습에서 떼어 내 처음 보는 데이터로 채점하기 위해
X_train, X_test, y_train, y_test = train_test_split(           # Q3 변수 이름 그대로
    X, y, test_size=0.2, stratify=y, random_state=7)           # 20% 봉인 · 품종 비율 유지 · 고정
print(f"Train: {len(X_train)}마리 · Test: {len(X_test)}마리")  # 8 : 2
print("전체 비율:", pd.Series(y).value_counts(normalize=True).round(3).to_dict())  # 품종 비율
print("Test 비율:", pd.Series(y_test).value_counts(normalize=True).round(3).to_dict())  # 거의 같음

Train: 273마리 · Test: 69마리
전체 비율: {'Adelie': 0.442, 'Gentoo': 0.36, 'Chinstrap': 0.199}
Test 비율: {'Adelie': 0.435, 'Gentoo': 0.362, 'Chinstrap': 0.203}


**결과** · 예측 정답은 273 · 69

- `Train: 273마리 · Test: 69마리` — 342 × 0.2 = 68.4 → 올림해 69마리를 봉인
- `전체 비율: {'Adelie': 0.442, 'Gentoo': 0.36, 'Chinstrap': 0.199}` — Adelie가 가장 많음
- `Test 비율: {'Adelie': 0.435, 'Gentoo': 0.362, 'Chinstrap': 0.203}` — `stratify=y` → 시험지 품종 비율도 거의 같음

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>Train (교과서)</td><td>모델이 패턴을 배움</td><td>fit · 교차검증 · 튜닝 전부</td></tr>
<tr><td>Test (봉인된 시험지)</td><td>학습 중 한 번도 안 보여 준 마지막 채점용</td><td>모든 선택이 끝난 뒤 1회</td></tr>
<tr><td>Validation (2장의 모의고사)</td><td>Train 안에서 떼어 반복 사용</td><td>모델 · 설정 고르기</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - test를 보고 모델을 바꾸면 test가 검증이 되어 점수가 부풀려짐 (데이터 누수)
> - 1-2회차 Q13의 island 문제와 본질이 같음

> ✅ **한 줄 정리** · 처음부터 8 : 2로 나눠 test를 봉인하고, test는 모든 선택이 끝난 뒤 마지막에 한 번만 쓴다.

**딥러닝 코드에서는**

- `train_set, val_set, test_set = random_split(...)` — test loader는 마지막에만
- Kaggle 비공개 리더보드 = 봉인된 시험지

### 1-3. `A` 진짜 시험 점수 · `fit(X_train)` · `predict(X_test)` <sub>원본 Q4</sub>

> 🎯 **처음 보는 test로 채점하면 1.0000보다 얼마나 낮아지는가?**

💡 **쉽게 말하면**

- 교과서로만 공부하고 **봉인된 시험지**로 채점.
- 이 점수가 새 펭귄에 대한 실력에 가까움.

**실행 흐름**

1. X_train으로 학습
2. X_test로 예측
3. acc_split 계산 · 착시 점수와 비교

**먼저 예측** · Q1의 100%보다 낮을까? 몇 %일까? — 답은 코드 실행 결과 아래에

In [4]:
# 왜 · 분할한 뒤의 점수가 cheat 점수와 어떻게 다른지 비교하기 위해
dt = DecisionTreeClassifier(random_state=42)                       # Q4 변수 이름 dt
dt.fit(X_train, y_train)                                           # 교과서(273마리)로만 공부
acc_split = accuracy_score(y_test, dt.predict(X_test))             # 봉인된 69마리로 채점
print(f"[Q1] 전체로 학습+평가 (cheat) : {acc_cheat:.4f}")          # 착시
print(f"[Q4] Train/Test 분리         : {acc_split:.4f}")           # 진짜 시험
print("틀린 마리 수:", int(round((1 - acc_split) * len(y_test))))  # 69마리 중

[Q1] 전체로 학습+평가 (cheat) : 1.0000
[Q4] Train/Test 분리         : 0.9130
틀린 마리 수: 6


**결과** · 예측 정답은 낮음 — 0.9130 (91.30%)

- `[Q1] 전체로 학습+평가 (cheat) : 1.0000` — 외운 문제 점수
- `[Q4] Train/Test 분리         : 0.9130` — 처음 보는 69마리 중 63마리 정답
- `틀린 마리 수: 6` — 69 - 63 = 6 · 1.0000과의 차이가 착시의 크기

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>train 점수</td><td>공부한 문제</td><td>과적합 진단용 (val과 비교)</td></tr>
<tr><td>test 점수</td><td>처음 보는 문제</td><td>최종 보고용</td></tr>
<tr><td>두 점수의 차이</td><td>외운 정도 (과적합)</td><td>차이가 크면 모델을 단순하게 (3장)</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - test 69마리면 1마리 = 0.0145 → 작은 차이로 결론 내리지 않기

> ✅ **한 줄 정리** · 처음 보는 test로 채점하면 1.0000이 0.9130으로 내려가며, 이 차이가 외워서 생긴 착시의 크기다.

**딥러닝 코드에서는**

- `model.eval()` + `torch.no_grad()`로 test loader 1회 평가 → 보고
- train acc 100% · val acc 91%면 과적합 신호

### 1-4. `A` seed만 바꾸면? · `random_state` · `for rs in` <sub>원본 Q5</sub>

> 🎯 **91.30%를 믿어도 되는가? seed만 바꾸면 시험 점수는 어떻게 되는가?**

💡 **쉽게 말하면**

- 시험지를 다시 뽑으면 **점수가 달라짐**.
- 한 번의 시험 점수에는 운이 섞여 있음.

**실행 흐름**

1. seed 5개로 매번 새로 분할
2. 매번 새 트리 학습
3. test 정확도 비교

**먼저 예측** · random_state를 바꾸면 정확도가 똑같이 나올까? — 답은 코드 실행 결과 아래에

In [5]:
# 왜 · 분할 seed만 바꿔도 점수가 흔들린다는 것을 확인하기 위해
random_states_to_try = [51, 92, 14, 71, 60]                                  # 원본 Q5의 seed 5개
accs = []                                                                    # seed마다 test 정확도
print(f"{'random_state':>12s}  {'test 정확도':>10s}")                        # 표 머리
for rs in random_states_to_try:                                              # 시험지를 다시 뽑음
    # 새 분할
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=rs)
    m = DecisionTreeClassifier(random_state=42).fit(Xtr, ytr)                # 새로 학습
    accs.append(accuracy_score(yte, m.predict(Xte)))                         # 새 시험지로 채점
    print(f"{rs:>12d}  {accs[-1]:>10.4f}")                                   # 한 줄씩
print(f"범위 {min(accs):.4f} ~ {max(accs):.4f} · 평균 {np.mean(accs):.4f}")  # 흔들리는 폭

random_state    test 정확도
          51      0.9420
          92      0.9420
          14      0.9565
          71      0.9710
          60      0.9275
범위 0.9275 ~ 0.9710 · 평균 0.9478


**결과** · 예측 정답은 달라짐 — 시험지가 바뀌기 때문

- `71      0.9710` — seed 71은 쉬운 시험지 · 69마리 중 2마리만 틀림
- `60      0.9275` — seed 60은 어려운 시험지 · 5마리 틀림
- `범위 0.9275 ~ 0.9710 · 평균 0.9478` — 같은 모델인데 seed만 바꿔도 0.04 넘게 흔들림 · seed 7의 0.9130은 더 낮음

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>한 번 분할</td><td>학습 1번 · 운이 섞임</td><td>코드 확인 · 마지막 test 1회</td></tr>
<tr><td>seed 여러 개 분할</td><td>흔들림을 보여 줌 · test를 여러 번 씀</td><td>흔들림 확인용 (선택에는 쓰지 않기)</td></tr>
<tr><td>교차검증 (2장)</td><td>train 안에서만 여러 번 · test 봉인 유지</td><td><strong>모델 · 설정 선택의 기본</strong></td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 좋은 seed를 골라 보고하면 운을 고른 것
> - seed를 바꿔 여러 번 test를 보는 것도 test를 여러 번 쓰는 셈 → 선택은 교차검증으로

> ✅ **한 줄 정리** · 한 번 나눈 test 점수는 seed에 따라 0.91 ~ 0.97로 흔들리므로, 모델 선택은 train 안의 교차검증으로 한다.

**딥러닝 코드에서는**

- seed 3개로 학습해 `평균 ± 표준편차` 보고
- val 데이터가 작으면 epoch마다 val 점수가 크게 흔들림

---
## 2. 모델 교체와 교차검증

> **이 장의 흐름**
>
> - 같은 분할에서 모델 셋을 test로 한 번 비교 (Q6)
> - 그다음 train 안에서 5등분해 K-Fold를 직접 구현 → `cross_val_score` 한 줄 → 셋을 CV로 다시 비교

<img src="images/pfm2_kfold.png" alt="Train 안에서 5등분 · 매번 1등분은 Validation · Test는 봉인" width="763">

**파이프라인 위치** · ④ 모델 학습 → ⑤ 교차검증

### 2-1. `B` 모델 셋 비교 · `LogisticRegression` · `KNeighborsClassifier` <sub>원본 Q6 · Part 3</sub>

> 🎯 **트리 · 로지스틱 · KNN을 같은 분할에서 바꿔 끼우면 순위는?**

💡 **쉽게 말하면**

- 모델을 바꿔도 `fit → predict → score`는 똑같음.
- 원리는 다음 회차에서 · 오늘은 바꿔 끼우기만.

**실행 흐름**

1. 모델 셋을 딕셔너리에
2. for문으로 fit → predict → 정확도
3. 순위 보기

**먼저 예측** · 셋 중 무엇이 제일 잘하고, 무엇이 제일 못할까? — 답은 코드 실행 결과 아래에

In [6]:
from sklearn.linear_model import LogisticRegression  # 클래스별 확률 → 가장 높은 것
from sklearn.neighbors import KNeighborsClassifier   # 가까운 이웃 k개 다수결

# 왜 · 같은 분할 · 같은 지표로 세 모델을 공정하게 비교하기 위해
models = {"DecisionTree": DecisionTreeClassifier(random_state=42),           # 질문을 던져 갈라 나감
          "LogisticRegression": LogisticRegression(max_iter=3000, random_state=42),  # 원본 Q6 설정
          "KNN": KNeighborsClassifier(n_neighbors=5)}                        # 이웃 5마리
print(f"{'모델':<20s}{'test 정확도':>10s}")                                  # 표 머리
for name, m in models.items():                                               # 같은 코드로 셋 다
    m.fit(X_train, y_train)                                                  # 같은 교과서
    print(f"{name:<20s}{accuracy_score(y_test, m.predict(X_test)):>10.4f}")  # 같은 시험지

모델                    test 정확도
DecisionTree            0.9130


LogisticRegression      0.9710
KNN                     0.7246


**결과** · 예측 정답은 로지스틱 1등 · KNN 꼴찌

- `DecisionTree            0.9130` — 1-3과 같은 값
- `LogisticRegression      0.9710` — 69마리 중 2마리만 틀림
- `KNN                     0.7246` — 가장 낮음 · 스케일링 없이 g(체중)이 거리를 지배

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>DecisionTree</td><td>조건 질문으로 갈라 나감</td><td>단위가 달라도 그대로 (파일별 6)</td></tr>
<tr><td>LogisticRegression</td><td>클래스별 확률 → 가장 높은 것</td><td>기준선 모델 (파일별 5)</td></tr>
<tr><td>KNN</td><td>가까운 이웃 k개 다수결</td><td>스케일링 후에 (파일별 3 · 보강 1)</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 이 순위도 test 69마리 한 번의 결과 → 2-4에서 교차검증으로 다시 확인
> - KNN이 낮은 이유는 모델이 나빠서가 아니라 전처리를 안 해서

> ✅ **한 줄 정리** · 같은 분할에서 모델만 바꿔 끼우면 비교할 수 있지만, test 한 번의 순위는 교차검증으로 다시 확인한다.

**딥러닝 코드에서는**

- 백본만 바꿔 같은 학습 코드로 비교
- 입력 정규화를 안 하면 어떤 모델은 크게 손해

### 2-2. `A` K-Fold 직접 구현 · `StratifiedKFold` · `kf.split` <sub>원본 Q7 · Part 4</sub>

> 🎯 **Train 안에서 5등분해 돌아가며 검증하는 K-Fold는 어떻게 동작하는가?**

💡 **쉽게 말하면**

- Train(교과서) · **Validation(모의고사, 5번 반복)** · Test(수능, 1회).
- 매번 1등분은 모의고사, 나머지 4등분으로 공부.

**실행 흐름**

1. StratifiedKFold 5등분 (섞어서)
2. fold마다 새 트리 학습 · 검증
3. fold_scores에 모아 평균 · 표준편차

**먼저 예측** · 5개 점수가 거의 같을까, 꽤 다를까? — 답은 코드 실행 결과 아래에

In [7]:
from sklearn.model_selection import StratifiedKFold  # 품종 비율 유지 5등분

# 왜 · 교차검증이 하는 일을 반복문으로 직접 풀어 이해하기 위해
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)                 # 섞어서 5등분
fold_scores = []  # fold마다 검증 정확도 (변수 이름 규약)
# (학습, 검증) 번호 쌍 5번
for k, (tr_idx, va_idx) in enumerate(kf.split(X_train, y_train), start=1):
    m = DecisionTreeClassifier(random_state=42)                                 # fold마다 새 모델
    m.fit(X_train[tr_idx], y_train[tr_idx])                                     # 4등분으로 공부
    # 1등분 모의고사
    fold_scores.append(accuracy_score(y_train[va_idx], m.predict(X_train[va_idx])))
    print(f"Fold {k}: {fold_scores[-1]:.4f} (학습 {len(tr_idx)} · 검증 {len(va_idx)})")  # fold마다
print(f"평균 {np.mean(fold_scores):.4f} · 표준편차 {np.std(fold_scores):.4f}")  # 요약

Fold 1: 0.9455 (학습 218 · 검증 55)
Fold 2: 0.9818 (학습 218 · 검증 55)
Fold 3: 0.9636 (학습 218 · 검증 55)
Fold 4: 0.9259 (학습 219 · 검증 54)
Fold 5: 0.9815 (학습 219 · 검증 54)
평균 0.9597 · 표준편차 0.0216


**결과** · 예측 정답은 꽤 다름 — 0.9259 ~ 0.9818

- `Fold 1: 0.9455 (학습 218 · 검증 55)` — 학습 4등분 218 + 검증 1등분 55
- `Fold 4: 0.9259 (학습 219 · 검증 54)` — 5로 나누어떨어지지 않아 뒤 두 fold는 54마리 · 가장 낮은 fold
- `평균 0.9597 · 표준편차 0.0216` — 모의고사 5번의 평균이 한 번 시험보다 믿을 만함

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>한 번 분할</td><td>검증 1번</td><td>빠른 확인</td></tr>
<tr><td><code>KFold</code></td><td>그냥 k등분</td><td>회귀 · 클래스가 고른 데이터</td></tr>
<tr><td><code>StratifiedKFold</code></td><td>등분마다 정답 비율 유지</td><td><strong>분류의 기본</strong></td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - fold마다 **새 모델**을 만들어야 함 → 앞 fold에서 배운 것이 남으면 안 됨
> - 검증 번호(`va_idx`)는 학습에 쓰지 않기

> ✅ **한 줄 정리** · K-Fold는 train을 k등분해 매번 1등분으로 검증하고 평균을 내며, test는 이 과정에서도 건드리지 않는다.

**딥러닝 코드에서는**

- 딥러닝은 학습이 비싸 K-Fold 대신 val 하나를 고정하는 경우가 많음
- 데이터가 적은 의료 영상은 5-Fold로 모델 5개를 학습

### 2-3. `A` `cross_val_score` 한 줄 · `cv=5` <sub>원본 Q8</sub>

> 🎯 **`cross_val_score` 한 줄은 Q7 for문과 같은 결과를 주는가?**

💡 **쉽게 말하면**

- for문 K-Fold를 **한 줄**로 줄인 함수.
- 다만 `cv=5`는 섞지 않고 순서대로 5등분 → Q7(섞음)과 숫자가 다름.

**실행 흐름**

1. cv=5로 한 줄 실행
2. Q7 평균과 비교
3. Q7과 같은 kf를 넣어 다시 실행

**먼저 예측** · Q7 평균과 똑같이 나올까? — 답은 코드 실행 결과 아래에

In [8]:
from sklearn.model_selection import cross_val_score  # 교차검증 한 줄

# 왜 · 같은 일을 cross_val_score 한 줄로 하고 결과를 비교하기 위해
tree = DecisionTreeClassifier(random_state=42)                    # 같은 설정
s5 = cross_val_score(tree, X_train, y_train, cv=5)                # 섞지 않는 5등분
print("cv=5 각 Fold:", s5.round(4), "평균", round(s5.mean(), 4))  # 원본 정답과 같은 형식
s_kf = cross_val_score(tree, X_train, y_train, cv=kf)             # Q7의 kf (섞은 5등분)
# 같은 분할
print("cv=kf 평균:", round(s_kf.mean(), 4), "· Q7 평균:", round(np.mean(fold_scores), 4))

cv=5 각 Fold: [0.9636 0.9273 0.9818 0.963  1.    ] 평균 0.9671
cv=kf 평균: 0.9597 · Q7 평균: 0.9597


**결과** · 예측 정답은 `cv=5`는 다름 (0.9671) · Q7과 같은 `kf`를 넣으면 같음

- `cv=5 각 Fold: [0.9636 0.9273 0.9818 0.963  1.    ] 평균 0.9671` — 섞지 않은 5등분 → Q7의 0.9597과 다름
- `cv=kf 평균: 0.9597 · Q7 평균: 0.9597` — 같은 등분을 넣으면 for문과 똑같음

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>for문 K-Fold (Q7)</td><td>과정을 다 보임</td><td>원리 이해 · fold별 처리가 다를 때</td></tr>
<tr><td><code>cross_val_score(cv=5)</code></td><td>한 줄 · 분류면 StratifiedKFold(섞지 않음)</td><td>빠른 비교</td></tr>
<tr><td><code>cross_val_score(cv=kf)</code></td><td>내가 만든 등분 그대로</td><td>다른 실험과 등분을 맞출 때</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - `cv=5`와 `cv=StratifiedKFold(shuffle=True)`는 등분이 다름 → 숫자를 비교할 때는 같은 cv
> - 데이터가 정렬돼 있으면(품종 순) 섞지 않은 등분이 치우칠 수 있음

> ✅ **한 줄 정리** · `cross_val_score`는 K-Fold for문을 한 줄로 줄인 것이며, 같은 등분(cv)을 넣어야 같은 숫자가 나온다.

**딥러닝 코드에서는**

- `KFold` 번호로 `Subset(dataset, idx)`를 만들어 fold마다 DataLoader 생성

### 2-4. `A` 세 모델을 교차검증으로 · `cross_val_score` · `.std()` <sub>원본 Q9</sub>

> 🎯 **Q6 순위는 교차검증으로 다시 비교해도 같은가?**

💡 **쉽게 말하면**

- 시험 한 번 대신 **모의고사 5번 평균**으로 줄 세우기.
- 표준편차는 점수가 얼마나 흔들리는지.

**실행 흐름**

1. Q6의 models 그대로
2. 모델마다 5-Fold 평균 · 표준편차
3. Q6 순위와 비교

**먼저 예측** · Q6이랑 순위가 같을까, 바뀔까? — 답은 코드 실행 결과 아래에

In [9]:
# 왜 · 한 번의 분할 대신 교차검증 평균 · 표준편차로 모델을 비교하기 위해
print(f"{'모델':<20s}{'CV 평균':>10s}{'CV 표준편차':>12s}")  # 표 머리
cv_mean = {}                                                 # 정리표용
for name, m in models.items():                               # Q6 딕셔너리 그대로
    s = cross_val_score(m, X_train, y_train, cv=5)           # train 안에서만 5번
    cv_mean[name] = s.mean()                                 # 평균 저장
    print(f"{name:<20s}{s.mean():>10.4f}{s.std():>12.4f}")   # 평균 · 흔들림

모델                       CV 평균     CV 표준편차
DecisionTree            0.9671      0.0241


LogisticRegression      0.9891      0.0145
KNN                     0.7801      0.0355


**결과** · 예측 정답은 순위는 같음 (로지스틱 > 트리 > KNN) · 점수는 test 한 번과 다름

- `DecisionTree            0.9671      0.0241` — test 0.9130보다 높음 → test 한 번이 어려운 시험지였음
- `LogisticRegression      0.9891      0.0145` — 1등 · 2등과 차이 0.022 > 표준편차 → 믿을 만한 1등
- `KNN                     0.7801      0.0355` — 꼴찌 · 스케일링 전이라

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>test 한 번 비교 (Q6)</td><td>69마리 · 운이 섞임 · test를 씀</td><td>쓰지 않기 (선택용으로는)</td></tr>
<tr><td>CV 평균 비교 (Q9)</td><td>273마리 전부가 한 번씩 검증 · test 봉인</td><td><strong>모델 선택</strong></td></tr>
<tr><td>CV 표준편차</td><td>fold마다 흔들림</td><td>평균 차이가 표준편차보다 작으면 같다고 봄</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - CV 점수도 고르는 데 쓴 점수 → 최종 보고는 test 1회 (3-3)

> ✅ **한 줄 정리** · 모델은 test가 아니라 train 안의 교차검증 평균 · 표준편차로 비교해 고른다.

**딥러닝 코드에서는**

- val 손실 · 정확도 곡선으로 모델 · 설정 비교 → test는 마지막

---
## 3. GridSearchCV와 최종 평가

> **이 장의 흐름**
>
> - max_depth를 직접 바꿔 CV 곡선을 그림 (Q10) → GridSearchCV로 자동화 (Q11)
> - 그다음 봉인한 test를 처음이자 마지막으로 열어 최종 점수 (Q12)

<img src="images/pfm2_depth.png" alt="max_depth별 CV 평균 ± 표준편차" width="610">

**파이프라인 위치** · ⑤ 교차검증 · 튜닝 → ⑥ 최종 평가

### 3-1. `A` `max_depth`를 직접 바꾸기 · `cross_val_score` <sub>원본 Q10</sub>

> 🎯 **`max_depth`를 1 → 2 → 3 → 4 → None으로 늘리면 점수는 계속 오르는가?**

💡 **쉽게 말하면**

- 깊이 = 트리가 던지는 **질문 횟수**.
- 너무 얕으면 덜 배우고, 너무 깊으면 외움.

**실행 흐름**

1. 깊이 5가지 후보
2. 후보마다 5-Fold 평균 · 표준편차
3. 표로 정리 (그래프는 3장 그림)

**먼저 예측** · 깊이를 늘리면 계속 오를까, 어디서 꺾일까? — 답은 코드 실행 결과 아래에

In [10]:
# 왜 · 깊이를 바꿔 가며 교차검증 점수를 비교하는 과정을 직접 해 보기 위해
rows = []                                                                  # (깊이, 평균, 표준편차)
for d in [1, 2, 3, 4, None]:                                               # 원본 Q10 후보
    model = DecisionTreeClassifier(max_depth=d, random_state=42)           # 깊이만 바꿈
    s = cross_val_score(model, X_train, y_train, cv=5)                     # 5-Fold
    rows.append((str(d), s.mean(), s.std()))                               # None도 글자로
depth_df = pd.DataFrame(rows, columns=["max_depth", "cv_mean", "cv_std"])  # 표
print(depth_df.round(4).to_string(index=False))                            # 깊이별 점수

max_depth  cv_mean  cv_std
        1   0.7803  0.0104
        2   0.9306  0.0311
        3   0.9378  0.0376
        4   0.9708  0.0246
     None   0.9671  0.0241


**결과** · 예측 정답은 4에서 가장 높고(0.9708), None(끝까지)은 조금 내려감

- `1   0.7803  0.0104` — 질문 1번 → 3품종을 둘로만 가름
- `4   0.9708  0.0246` — 가장 높음
- `None   0.9671  0.0241` — 끝까지 갈라 외우면 오히려 조금 내려감 (2-4의 트리와 같은 값)

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>얕은 트리 (1 · 2)</td><td>덜 배움 (과소적합)</td><td>train · val 둘 다 낮을 때 깊이를 늘림</td></tr>
<tr><td>적당한 깊이 (3 · 4)</td><td>val 최고</td><td>고를 값</td></tr>
<tr><td>끝까지 (None)</td><td>외움 (과적합) · train 1.0</td><td>train ≫ val일 때 깊이를 줄임</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 깊이별 점수 차이(0.9708 vs 0.9671)가 표준편차보다 작음 → 4와 None의 차이는 크지 않음

> ✅ **한 줄 정리** · 깊이를 늘리면 처음엔 오르다 끝까지 가면 외워서 내려가므로, 교차검증 평균이 가장 높은 깊이를 고른다.

**딥러닝 코드에서는**

- 층 수 · 채널 수 · epoch도 같은 곡선 → val 최고점에서 멈춤 (early stopping)

### 3-2. `A` GridSearchCV로 자동화 · `GridSearchCV` · `best_params_` · `cv_results_` <sub>원본 Q11</sub>

> 🎯 **`GridSearchCV`는 무엇을 대신해 주고, 무엇은 해 주지 않는가?**

💡 **쉽게 말하면**

- Q10의 for문을 대신 돌리는 **탐색기**.
- 후보는 사람이 정함 → 후보가 틀리면 아무리 돌려도 소용없음.

**실행 흐름**

1. param_grid에 후보 넣기
2. GridSearchCV(cv=5)로 fit
3. 최적 깊이 · 점수 · 전체 표 보기

**먼저 예측** · 최적 max_depth는 몇일까? (Q10 표를 보고) — 답은 코드 실행 결과 아래에

In [11]:
from sklearn.model_selection import GridSearchCV  # 후보 × 교차검증 자동화

# 왜 · 직접 비교한 과정을 GridSearchCV로 자동화하기 위해
param_grid = {"max_depth": [1, 2, 3, 4, None]}                            # 사람이 정한 후보
grid = GridSearchCV(DecisionTreeClassifier(random_state=42), param_grid,  # 탐색할 모델 · 후보
                    cv=5, scoring="accuracy", return_train_score=True)    # 훈련 점수도 기록
grid.fit(X_train, y_train)  # 후보 5 × fold 5 = 25번 학습 + 최적값으로 train 전체 재학습
print("최적 파라미터:", grid.best_params_)                                # Q10에서 가장 높던 깊이
print(f"최적 CV 점수: {grid.best_score_:.4f}")                            # 고를 때 쓴 점수
cols = ["param_max_depth", "mean_train_score", "mean_test_score", "rank_test_score"]  # 볼 열
res = pd.DataFrame(grid.cv_results_)[cols]                                # 후보별 결과 표
print(res.round(4).to_string(index=False))                                # 훈련 vs 검증

최적 파라미터: {'max_depth': 4}
최적 CV 점수: 0.9708
param_max_depth  mean_train_score  mean_test_score  rank_test_score
              1            0.7903           0.7803                5
              2            0.9670           0.9306                4
              3            0.9835           0.9378                3
              4            0.9973           0.9708                1
           None            1.0000           0.9671                2


**결과** · 예측 정답은 4

- `최적 파라미터: {'max_depth': 4}` — Q10 표와 같은 답
- `최적 CV 점수: 0.9708` — Q10의 깊이 4 평균과 같음 · 원본 Q12의 0.9708과도 같음
- `None            1.0000           0.9671                2` — 끝까지 가면 훈련 1.0000 · 검증 0.9671 → 외운 만큼 차이

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>for문 (Q10)</td><td>과정이 보임</td><td>후보가 적고 원리를 볼 때</td></tr>
<tr><td><code>GridSearchCV</code></td><td>후보 × fold 자동 · 최적 모델 재학습까지</td><td><strong>튜닝의 기본</strong></td></tr>
<tr><td><code>RandomizedSearchCV</code></td><td>후보에서 무작위로 n개</td><td>후보가 아주 많을 때</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - GridSearchCV는 '설계자'가 아니라 '탐색기' → 후보 범위를 잘못 잡으면 소용없음
> - `best_score_`는 고르는 데 쓴 점수라 낙관적 → 최종 성적이 아님

> ✅ **한 줄 정리** · `GridSearchCV`는 사람이 정한 후보를 교차검증으로 비교해 최적값을 고르고 train 전체로 다시 학습한다.

**딥러닝 코드에서는**

- Optuna · Ray Tune — 학습률 · 배치 크기 탐색기 (후보 범위는 사람이 정함)

### 3-3. `A` 최종 홀드아웃 평가 · `best_estimator_` · `final_acc` <sub>원본 Q12</sub>

> 🎯 **봉인한 test는 언제 열고, CV 점수와 비슷하게 나오는가?**

💡 **쉽게 말하면**

- 모든 선택이 끝난 뒤 **수능을 딱 한 번**.
- 이 점수를 보고 다시 고르지 않음.

**실행 흐름**

1. best_estimator_ 꺼내기
2. X_test로 처음이자 마지막 예측
3. CV 점수와 비교

**먼저 예측** · CV 점수(0.9708)와 최종 test 점수는 비슷할까? — 답은 코드 실행 결과 아래에

In [12]:
# 왜 · 고를 때 쓰지 않은 test로 마지막에 한 번만 채점하기 위해
best_model = grid.best_estimator_  # 깊이 4로 train 전체를 다시 학습한 모델
final_acc = accuracy_score(y_test, best_model.predict(X_test))        # test 첫 사용
print(f"최적 파라미터        : {grid.best_params_}")                  # 고른 설정
print(f"최적 CV 점수 (train) : {grid.best_score_:.4f}")               # 고를 때 쓴 점수
print(f"최종 test 점수       : {final_acc:.4f}")                      # 보고할 점수
print(f"차이                 : {grid.best_score_ - final_acc:+.4f}")  # 낙관 정도

최적 파라미터        : {'max_depth': 4}
최적 CV 점수 (train) : 0.9708
최종 test 점수       : 0.9130
차이                 : +0.0578


**결과** · 예측 정답은 0.9130 — CV보다 0.0578 낮음 · 튜닝 전 1-3과 같은 점수

- `최적 CV 점수 (train) : 0.9708` — 고를 때 쓴 점수
- `최종 test 점수       : 0.9130` — 보고할 점수 · 튜닝 전(1-3)과 같은 63마리 정답
- `차이                 : +0.0578` — 고를 때 쓴 점수는 낙관적 · 튜닝이 test를 올려 준다는 보장도 없음

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>CV 점수</td><td>고를 때 쓴 점수 · 낙관적</td><td>모델 · 설정 선택</td></tr>
<tr><td>test 점수</td><td>한 번도 안 쓴 데이터</td><td><strong>최종 보고 (1회)</strong></td></tr>
<tr><td>test를 보고 다시 고르기</td><td>test가 검증이 됨</td><td>쓰지 않기</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - 튜닝했는데 test가 그대로라고 다시 튜닝하면 test가 무효 → 이 0.9130을 그대로 보고
> - 69마리 test에서는 깊이 4와 None의 차이가 보이지 않을 수 있음

> ✅ **한 줄 정리** · 모든 선택이 끝난 뒤 test를 한 번만 열어 보고하며, CV 점수보다 낮게 나와도 그대로 보고한다.

**딥러닝 코드에서는**

- `best.pt`를 불러 test loader로 1회 평가 → 논문 · 보고서의 숫자

---
## 4. 종합 실습 · 오늘 정리표 채우기 · KNN으로 같은 흐름

> **이 장의 흐름**
>
> - 1 ~ 3장의 숫자로 원본 정리표의 빈칸을 채움
> - 그다음 KNN의 `n_neighbors`를 GridSearchCV로 골라 test 1회까지 같은 순서로

<img src="images/pfm2_practice.png" alt="오늘 정리표 · 같은 데이터 채점부터 최종 test까지" width="677">

**파이프라인 위치** · ② 분할 → ⑤ 교차검증 · 튜닝 → ⑥ 최종 평가

> 🎯 **트리 흐름을 그대로 KNN에 옮기면 최적 k와 최종 test 점수는?**

💡 **쉽게 말하면**

- 순서는 같음 · **분할 → CV로 고르기 → test 1회**.
- 모델과 후보만 바뀜.

**실행 흐름**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">단계</th><th style="text-align: left;">하는 일</th><th style="text-align: left;">쓰는 문법</th></tr>
</thead>
<tbody>
<tr><td>1</td><td>원본 정리표 빈칸 채우기</td><td>1-1 · 1-3 · 2-4 · 3-2 · 3-3</td></tr>
<tr><td>2</td><td>KNN <code>n_neighbors</code> 후보 정하기</td><td>3-2</td></tr>
<tr><td>3</td><td>GridSearchCV로 train 안에서 고르기</td><td>3-2</td></tr>
<tr><td>4</td><td>test 1회로 최종 점수</td><td>3-3</td></tr>
</tbody></table>

**먼저 예측** · KNN도 k를 잘 고르면 트리(0.9130)를 넘을까? — 답은 코드 실행 결과 아래에

In [13]:
# 1단계: 원본 '오늘 정리' 표 채우기
# 왜 · 방법마다 점수를 한 표에 모아 어떤 평가가 믿을 만한지 비교하기 위해
names = ["전체로 학습+평가 (cheat)", "Train/Test 분리",  # 원본 표의 방법 6개
         "5-Fold 평균 (DecisionTree)", "5-Fold 평균 (LogisticRegression)",
         "GridSearchCV 최적 CV 점수", "최종 홀드아웃 test 점수"]
vals = [acc_cheat, acc_split, cv_mean["DecisionTree"],   # 앞 장의 변수 그대로
        cv_mean["LogisticRegression"], grid.best_score_, final_acc]
summary = pd.DataFrame({"방법": names, "정확도": vals})  # 빈칸이 채워진 표
print(summary.round(4).to_string(index=False))           # 빈칸이 채워진 표

                            방법    정확도
             전체로 학습+평가 (cheat) 1.0000
                 Train/Test 분리 0.9130
      5-Fold 평균 (DecisionTree) 0.9671
5-Fold 평균 (LogisticRegression) 0.9891
         GridSearchCV 최적 CV 점수 0.9708
               최종 홀드아웃 test 점수 0.9130

**실행 흐름**

1. train 안에서 GridSearchCV로 KNN의 k 고르기
2. 고른 모델로 test를 한 번만 채점
3. 앞 표의 다른 방법과 비교

In [14]:
# 2 · 3단계: KNN의 k를 train 안에서 고르기 (test는 아직 봉인)
# 왜 · test를 고르는 데 쓰면 점수가 낙관적으로 나오므로 train 안에서만 고르기 위해
# 후보 6개
knn_grid = GridSearchCV(KNeighborsClassifier(), {"n_neighbors": [1, 3, 5, 7, 9, 15]}, cv=5)
knn_grid.fit(X_train, y_train)                                                   # 6 × 5 = 30번 학습
print("KNN 최적 k:", knn_grid.best_params_, f"· CV {knn_grid.best_score_:.4f}")  # 고른 k


# 4단계: test 1회
# 왜 · 선택이 끝난 뒤 test를 한 번만 써서 공정한 최종 점수를 얻기 위해
knn_final = accuracy_score(y_test, knn_grid.best_estimator_.predict(X_test))  # 처음이자 마지막
# 비교 (선택은 이미 끝남)
print(f"KNN 최종 test: {knn_final:.4f} · 트리 최종 test: {final_acc:.4f}")

KNN 최적 k:

 {'n_neighbors': 1} · CV 0.8314
KNN 최종 test: 0.7971 · 트리 최종 test: 0.9130


**결과** · 예측 정답은 못 넘음 — KNN은 스케일링 없이는 0.8 근처

- `전체로 학습+평가 (cheat) 1.0000` — 착시 점수
- `Train/Test 분리 0.9130` — 한 번 시험
- `5-Fold 평균 (LogisticRegression) 0.9891` — CV 1등 모델
- `GridSearchCV 최적 CV 점수 0.9708` — 고를 때 쓴 점수
- `최종 홀드아웃 test 점수 0.9130` — 보고할 점수
- `KNN 최적 k: {'n_neighbors': 1} · CV 0.8314` — k=1이 최적 · 그래도 트리 CV보다 한참 낮음
- `KNN 최종 test: 0.7971 · 트리 최종 test: 0.9130` — k만 고쳐서는 부족 → 단위를 맞추는 스케일링이 필요 (파일별 3)

**차이 · 언제 쓰나**

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">방법</th><th style="text-align: left;">차이</th><th style="text-align: left;">언제 쓰나</th></tr>
</thead>
<tbody>
<tr><td>트리 흐름</td><td>깊이 4 · CV 0.9708 · test 0.9130</td><td>단위가 달라도 됨</td></tr>
<tr><td>KNN 흐름</td><td>k=1 · CV 0.8314 · test 0.7971</td><td>스케일링 뒤에 다시 (보강 1)</td></tr>
<tr><td>로지스틱 (CV 0.9891)</td><td>튜닝 없이도 CV 1등</td><td>튜닝 전에 기준선으로 먼저</td></tr>
</tbody></table>

> ⚠️ **주의**
>
> - KNN 결과를 보고 트리 대신 KNN을 고르거나 그 반대로 다시 고르는 것은 test로 고르는 셈
> - 모델 후보 비교도 CV에서 끝내고 test는 마지막 1회

> ✅ **한 줄 정리** · 모델이 바뀌어도 분할 → train 안의 CV로 고르기 → test 1회라는 순서는 그대로다.

**딥러닝 코드에서는**

- 백본 · 학습률을 val로 고르고 `best.pt`로 test 1회 — 같은 순서

---
## 그래서 오늘 무엇을 배웠는가?

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">학습 목표</th><th style="text-align: left;">할 수 있게 된 것</th><th style="text-align: left;">확인 위치</th></tr>
</thead>
<tbody>
<tr><td>1. 착시</td><td>전체 채점 1.0000 → test 0.9130 (잎 14개로 외움)</td><td>1-1 · 1-3</td></tr>
<tr><td>2. 분할 · 봉인</td><td>273 · 69 · stratify · seed마다 0.9275 ~ 0.9710</td><td>1-2 · 1-4</td></tr>
<tr><td>3. K-Fold</td><td>for문 평균 0.9597 = <code>cross_val_score(cv=kf)</code> · <code>cv=5</code>는 0.9671</td><td>2-2 · 2-3</td></tr>
<tr><td>4. GridSearchCV</td><td>후보 1 ~ None 중 깊이 4 · CV 0.9708 · 후보는 사람이</td><td>3-1 · 3-2</td></tr>
<tr><td>5. test 1회</td><td>최종 0.9130 · CV보다 0.0578 낮아도 그대로 보고</td><td>3-3 · 4</td></tr>
</tbody></table>

### 주제별 한 줄 · 반드시 기억할 것

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">등급</th><th style="text-align: left;">주제</th><th style="text-align: left;">반드시 기억할 것</th><th style="text-align: left;">딥러닝 코드에서</th></tr>
</thead>
<tbody>
<tr><td><code>A</code></td><td>착시</td><td>같은 데이터로 채점하면 부풀려짐</td><td>train acc만 보지 않기</td></tr>
<tr><td><code>A</code></td><td>test 봉인</td><td>모든 선택 뒤 1회</td><td>test loader는 마지막</td></tr>
<tr><td><code>A</code></td><td>K-Fold</td><td>train 안에서 k번 검증 · 평균</td><td>fold별 모델</td></tr>
<tr><td><code>A</code></td><td>cross_val_score</td><td>K-Fold 한 줄 · 같은 cv면 같은 숫자</td><td><code>Subset</code> + DataLoader</td></tr>
<tr><td><code>A</code></td><td>GridSearchCV</td><td>후보 × CV 탐색기 · 재학습까지</td><td>Optuna</td></tr>
<tr><td><code>A</code></td><td>best_score_</td><td>고를 때 쓴 점수 → 낙관적</td><td>best val acc</td></tr>
<tr><td><code>B</code></td><td>stratify</td><td>분할마다 정답 비율 유지</td><td>클래스별 샘플링</td></tr>
<tr><td><code>B</code></td><td>max_depth</td><td>질문 횟수 · 과소 vs 과적합</td><td>층 수 · epoch</td></tr>
</tbody></table>

---
## 기억 확인 문제

위 내용을 **보지 않고** 답한 뒤, ✅ 한 줄 정리와 비교해 ○ (정확) · △ (일부) · × (모름)를 표시. △ · ×인 섹션만 다시 보기.

### 1. 질문에 답하기

*섹션 → 🎯 질문*

- **1-1** · 전체로 학습 · 채점하면 몇 %이고 왜 믿으면 안 되나?
- **1-2** · 342마리 8 : 2 분할 결과는?
- **1-3** · 처음 보는 test 점수는?
- **1-4** · seed만 바꾸면 test 점수는?
- **2-1** · Q6 test 한 번 비교의 순위는?
- **2-2** · K-Fold에서 Validation과 Test의 차이는?
- **2-3** · `cross_val_score(cv=5)`가 Q7과 다른 이유는?
- **2-4** · 모델 선택은 무엇으로 하나?
- **3-1** · max_depth를 늘리면?
- **3-2** · GridSearchCV가 해 주지 않는 것은?
- **3-3** · CV 점수와 test 점수의 관계는?

#### 내 답 · 이 셀을 더블클릭해서 작성

**1-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**1-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**1-3** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**1-4** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-3** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**2-4** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**3-1** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**3-2** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

**3-3** · (여기에 답 작성) · ○ △ × 중 하나만 남기기

### 2. 코드로 답하기

아래 빈 셀에 직접 작성 후 실행.

1. `LogisticRegression(max_iter=3000, C=c)`의 `C`를 `[0.01, 0.1, 1]`에서 GridSearchCV(cv=5)로 골라 최적 C 출력
2. 고른 모델로 test 정확도를 소수 넷째 자리까지 출력

In [15]:
# 코드 문제 직접 작성

### 3. 변형 문제 · 조건을 바꿔 다시 풀기

1. 2-2에서 `n_splits=10`으로 바꾸면 평균과 표준편차는 어떻게 변할까?
2. 3-2의 후보를 `[1, 2]`로만 주면 최적 깊이와 최종 test 점수는? ('탐색기' 확인)
3. 1-2에서 `stratify=y`를 빼고 seed를 0 · 1 · 2로 바꾸면 Test의 Chinstrap 비율은 여전히 0.2 근처일까?

In [16]:
# 변형 문제 직접 작성

### 4. 모범 답안 · 1 ~ 3을 다 푼 뒤에 보기

#### 질문 답

*섹션 → 답*

- **1-1** · 1.0000 · 외운 문제로 시험 본 셈
- **1-2** · Train 273 · Test 69 (stratify로 품종 비율 유지)
- **1-3** · 0.9130 (69마리 중 63마리)
- **1-4** · 0.9275 ~ 0.9710으로 흔들림 → 선택은 CV로
- **2-1** · 로지스틱 0.9710 &gt; 트리 0.9130 &gt; KNN 0.7246
- **2-2** · Validation은 train 안 모의고사(반복) · Test는 수능(1회)
- **2-3** · cv=5는 섞지 않은 등분 · Q7은 섞은 등분
- **2-4** · train 안의 CV 평균 · 표준편차
- **3-1** · 오르다가 끝까지 가면 외워서 조금 내려감 (4가 최고)
- **3-2** · 후보 범위 정하기 (탐색기일 뿐)
- **3-3** · CV는 고를 때 쓴 점수라 낙관적 · test가 낮아도 그대로 보고

#### 코드 문제 답

```python
lr = LogisticRegression(max_iter=3000, random_state=42)
lr_grid = GridSearchCV(lr, {"C": [0.01, 0.1, 1]}, cv=5)
lr_grid.fit(X_train, y_train)
print(lr_grid.best_params_)  # {'C': 1}
print(f"{accuracy_score(y_test, lr_grid.best_estimator_.predict(X_test)):.4f}")  # 0.9710
```

#### 변형 문제 답

*문제 → 답*

- **1** · 평균 0.9708 · 표준편차 0.0273 — 검증 덩어리가 작아져(27 ~ 28마리) 흔들림이 커짐
- **2** · 최적 2 · test 0.9130 — 후보 밖의 4는 찾지 못함
- **3** · 아님 — 0.174 · 0.145 · 0.232로 흔들림 (seed 7은 우연히 0.203) · stratify가 비율을 지켜 줌

---
## 오답 노트

틀린 문제는 정답을 베끼지 말고 **원인**을 적음. 다음 복습 때 이 표의 "재발 방지 질문"부터 스스로에게 물어보기.

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">틀린 내용</th><th style="text-align: left;">틀린 이유</th><th style="text-align: left;">올바른 원칙</th><th style="text-align: left;">재발 방지 질문</th></tr>
</thead>
<tbody>
<tr><td>(예) test 점수가 낮아 max_depth를 다시 고름</td><td>test를 검증처럼 씀</td><td>선택은 CV, test는 마지막 1회</td><td>이 숫자를 보고 무언가를 바꿨나?</td></tr>
<tr><td></td><td></td><td></td><td></td></tr>
<tr><td></td><td></td><td></td><td></td></tr>
</tbody></table>

---
## 면접 30초 답변

**Q1. 교차검증은 왜 하나요?**
> - 한 번 나눈 시험 점수는 어떤 데이터가 시험에 들어갔는지에 따라 흔들립니다.
> - 펭귄에서도 seed만 바꿔 0.93에서 0.97까지 달라졌습니다.
> - 그래서 train 안에서 k번 나눠 검증하고 평균과 표준편차로 모델을 비교하며, test는 봉인해 둡니다.

**Q2. GridSearchCV의 best_score_를 최종 성능으로 보고해도 되나요?**
> - 안 됩니다.
> - best_score_는 여러 후보 중 가장 좋은 것을 고를 때 쓴 점수라 낙관적입니다.
> - 고른 모델을 한 번도 쓰지 않은 test로 한 번 평가한 점수를 보고합니다.
> - 펭귄에서는 CV 0.9708, test 0.9130이었습니다.

**Q3. Validation과 Test의 차이는 무엇인가요?**
> - Validation은 train 안에서 떼어 모델과 설정을 고르는 데 여러 번 쓰는 모의고사입니다.
> - Test는 모든 선택이 끝난 뒤 한 번만 쓰는 수능입니다.
> - test를 보고 다시 고르면 test가 검증이 되어 성능이 부풀려집니다.

---
## 최종 요약표

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">함수·문법</th><th style="text-align: left;">소속</th><th style="text-align: left;">역할</th><th style="text-align: left;">코드에서 사용한 위치</th></tr>
</thead>
<tbody>
<tr><td><code>pd.read_csv</code> · <code>dropna(subset=)</code></td><td>pandas</td><td>펭귄 읽기 · 측정값 결측 제거</td><td>0 <code>peng_raw.dropna(subset=meas)</code></td></tr>
<tr><td><code>DecisionTreeClassifier</code></td><td>sklearn.tree</td><td>결정 트리</td><td>1-1 <code>DecisionTreeClassifier(random_state=42)</code></td></tr>
<tr><td><code>get_n_leaves()</code></td><td>결정 트리 메서드</td><td>잎 개수</td><td>1-1 <code>dt_cheat.get_n_leaves()</code></td></tr>
<tr><td><code>train_test_split</code></td><td>sklearn.model_selection</td><td>8 : 2 분할</td><td>1-2 <code>train_test_split(X, y, test_size=0.2, …)</code></td></tr>
<tr><td><code>accuracy_score</code></td><td>sklearn.metrics</td><td>정확도</td><td>1-3 <code>accuracy_score(y_test, dt.predict(X_test))</code></td></tr>
</tbody></table>

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">함수·문법</th><th style="text-align: left;">소속</th><th style="text-align: left;">역할</th><th style="text-align: left;">코드에서 사용한 위치</th></tr>
</thead>
<tbody>
<tr><td><code>LogisticRegression</code> · <code>KNeighborsClassifier</code></td><td>sklearn</td><td>모델 교체</td><td>2-1 <code>models = {…}</code></td></tr>
<tr><td><code>StratifiedKFold</code> · <code>split()</code></td><td>sklearn.model_selection</td><td>비율 유지 k등분</td><td>2-2 <code>kf.split(X_train, y_train)</code></td></tr>
<tr><td><code>cross_val_score</code></td><td>sklearn.model_selection</td><td>교차검증 한 줄</td><td>2-3 <code>cross_val_score(…, cv=5)</code></td></tr>
<tr><td><code>GridSearchCV</code></td><td>sklearn.model_selection</td><td>후보 × CV 탐색</td><td>3-2 <code>GridSearchCV(…, param_grid, cv=5)</code></td></tr>
<tr><td><code>best_params_</code> · <code>best_score_</code> · <code>best_estimator_</code></td><td>GridSearchCV 속성</td><td>최적값 · 점수 · 모델</td><td>3-3 <code>best_model = grid.best_estimator_</code></td></tr>
</tbody></table>

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">함수·문법</th><th style="text-align: left;">소속</th><th style="text-align: left;">역할</th><th style="text-align: left;">코드에서 사용한 위치</th></tr>
</thead>
<tbody>
<tr><td><code>cv_results_</code></td><td>GridSearchCV 속성</td><td>후보별 훈련 · 검증 점수</td><td>3-2 <code>pd.DataFrame(grid.cv_results_)</code></td></tr>
</tbody></table>

### 할 것 · 하지 말 것

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">✅ 할 것</th><th style="text-align: left;">⚠️ 하지 말 것</th></tr>
</thead>
<tbody>
<tr><td>test는 처음에 떼어 봉인</td><td>같은 데이터로 학습 · 채점</td></tr>
<tr><td>선택은 train 안의 CV로</td><td>test 점수를 보고 다시 고르기</td></tr>
<tr><td>fold마다 새 모델</td><td>좋은 seed 골라 보고</td></tr>
<tr><td>후보 범위는 직접 넓게</td><td><code>best_score_</code>를 최종 성적으로</td></tr>
</tbody></table>

### 헷갈리는 짝 한눈에 보기

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">짝</th><th style="text-align: left;">차이</th><th style="text-align: left;">결론</th></tr>
</thead>
<tbody>
<tr><td>Validation vs Test</td><td>모의고사(반복) vs 수능(1회)</td><td>고르기는 Validation</td></tr>
<tr><td><code>cv=5</code> vs <code>cv=kf</code></td><td>섞지 않은 등분 vs 내가 만든 등분</td><td>비교는 같은 cv</td></tr>
<tr><td><code>best_score_</code> vs test 점수</td><td>고를 때 쓴 점수 vs 처음 보는 점수</td><td>보고는 test</td></tr>
<tr><td>for문 K-Fold vs <code>cross_val_score</code></td><td>과정이 보임 vs 한 줄</td><td>결과는 같음 (같은 cv)</td></tr>
</tbody></table>

---
## 다음 복습일

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">언제</th><th style="text-align: left;">할 일</th><th style="text-align: left;">걸리는 시간</th></tr>
</thead>
<tbody>
<tr><td>내일</td><td>기억 확인 문제 1을 자료 없이 다시 답하고 오답 노트 갱신</td><td>10분</td></tr>
<tr><td>3일 뒤</td><td>변형 문제 3개 · 2-2 K-Fold for문을 빈 셀에 직접</td><td>20분</td></tr>
<tr><td>7일 뒤</td><td>4장 종합 실습을 로지스틱 C로 바꿔 처음부터 다시</td><td>25분</td></tr>
<tr><td>14일 뒤</td><td>오답 노트에 남은 문제만 · 파일별 3(전처리) 시작 전 ✅ 줄만 읽기</td><td>10분</td></tr>
</tbody></table>

---
## 부록

### 이 노트북을 만든 방법

- 원본 문제 노트북(Q1 ~ Q12 · TODO 빈 셀)을 직접 풀어 1 ~ 3장 · 11개 섹션으로 정리
- seed · 설정은 같은 폴더의 정답 코드와 맞춤 (분할 7 · 모델 42) · 정답 코드는 문제를 푼 뒤에 비교
- 원본 '오늘 정리' 표 빈칸은 4장에서 실제 값으로 채움 · Q10 그래프는 3장 그림으로
- 모든 코드는 실행한 출력과 함께 저장 · 형식 v2.1 (FORMAT_V2.md 1 ~ 7절)

### 읽는 법

*표시 → 의미*

- **`A` · `B`** · **A** 딥러닝 코드에 바로 나오는 필수 · **B** 알아두면 되는 것
- **🎯** · 이번 섹션에서 답해야 할 질문
- **💡** · 쉬운 설명 · 비유
- **⚠️** · 자주 하는 실수
- **✅** · 기억할 결론 = 🎯 질문의 답

- **섹션 앞부분** · 🎯 질문 → 💡 쉬운 설명 → 실행 흐름 → 먼저 예측 → 코드
- **섹션 뒷부분** · 결과(예측 정답 · 출력 줄) → 차이 · 언제 쓰나 → ⚠️ 주의 → ✅ 한 줄 정리
- **코드 주석** · 코드 위는 이 단계에서 하는 일, 코드 옆은 그 줄이 필요한 이유 (줄마다 · 열 맞춤)

### 원본 → 섹션 대응표

<table width="100%" style="width:100%; table-layout:auto">
<thead><tr><th style="text-align: left;">원본</th><th style="text-align: left;">내용</th><th style="text-align: left;">이 노트북</th></tr>
</thead>
<tbody>
<tr><td>셀 0 ~ 2</td><td>도입 · 실습 규칙 · 변수 이름 규약</td><td>0. 원본 문제</td></tr>
<tr><td>셀 3</td><td>준비 (342마리)</td><td>0. 준비 코드</td></tr>
<tr><td>셀 4 · 5 (Q1)</td><td>전체로 학습 · 채점</td><td>1-1</td></tr>
<tr><td>셀 6 ~ 8 (Q2 · Q3)</td><td>봉인 규칙 · 8 : 2 분할</td><td>1-2</td></tr>
<tr><td>셀 9 · 10 (Q4)</td><td>진짜 시험 점수</td><td>1-3</td></tr>
<tr><td>셀 11 · 12 (Q5)</td><td>seed 바꾸기</td><td>1-4</td></tr>
<tr><td>셀 13 · 14 (Q6)</td><td>모델 셋 비교</td><td>2-1</td></tr>
<tr><td>셀 15 · 16 (Q7)</td><td>K-Fold 직접 구현</td><td>2-2</td></tr>
<tr><td>셀 17 · 18 (Q8)</td><td>cross_val_score</td><td>2-3</td></tr>
<tr><td>셀 19 · 20 (Q9)</td><td>CV로 세 모델</td><td>2-4</td></tr>
<tr><td>셀 21 · 22 (Q10)</td><td>max_depth 곡선</td><td>3-1 · 3장 그림</td></tr>
<tr><td>셀 23 · 24 (Q11)</td><td>GridSearchCV</td><td>3-2</td></tr>
<tr><td>셀 25 · 26 (Q12)</td><td>최종 홀드아웃</td><td>3-3</td></tr>
<tr><td>셀 27</td><td>오늘 정리 표</td><td>4. 종합 실습 1단계</td></tr>
</tbody></table>

### 전체 코드

1 ~ 4장 코드를 장별 셀로 모음. **위에서부터 이 셀들만 실행해도 전체가 동작**하며, 각 블록 머리 주석의 번호가 섹션.

**실행 흐름**

1. 1-1 cheat 점수
2. 1-2 · 1-3 분할 후 점수 · seed 흔들림
3. 1-4 세 모델 비교

In [17]:
from sklearn.metrics import accuracy_score            # 정확도
from sklearn.tree import DecisionTreeClassifier       # 결정 트리
from sklearn.model_selection import train_test_split  # 학습 · 시험 분할


# ---------- 1-1 ----------

# 왜 · 학습에 쓴 데이터로 채점하면 점수가 부풀려지는 문제를 먼저 보이기 위해
dt_cheat = DecisionTreeClassifier(random_state=42)    # 원본 정답과 같은 seed
dt_cheat.fit(X, y)                                    # 342마리 전부로 공부
acc_cheat = accuracy_score(y, dt_cheat.predict(X))    # 공부한 그 문제로 채점
print(f"전체로 학습 + 전체로 채점: {acc_cheat:.4f}")  # 착시 점수
print("트리 잎 개수:", dt_cheat.get_n_leaves())       # 끝까지 갈라 외운 정도


# ---------- 1-2 ----------

# 왜 · 평가용 데이터를 학습에서 떼어 내 처음 보는 데이터로 채점하기 위해
X_train, X_test, y_train, y_test = train_test_split(           # Q3 변수 이름 그대로
    X, y, test_size=0.2, stratify=y, random_state=7)           # 20% 봉인 · 품종 비율 유지 · 고정
print(f"Train: {len(X_train)}마리 · Test: {len(X_test)}마리")  # 8 : 2
print("전체 비율:", pd.Series(y).value_counts(normalize=True).round(3).to_dict())  # 품종 비율
print("Test 비율:", pd.Series(y_test).value_counts(normalize=True).round(3).to_dict())  # 거의 같음


# ---------- 1-3 ----------
# 왜 · 분할한 뒤의 점수가 cheat 점수와 어떻게 다른지 비교하기 위해
dt = DecisionTreeClassifier(random_state=42)                       # Q4 변수 이름 dt
dt.fit(X_train, y_train)                                           # 교과서(273마리)로만 공부
acc_split = accuracy_score(y_test, dt.predict(X_test))             # 봉인된 69마리로 채점
print(f"[Q1] 전체로 학습+평가 (cheat) : {acc_cheat:.4f}")          # 착시
print(f"[Q4] Train/Test 분리         : {acc_split:.4f}")           # 진짜 시험
print("틀린 마리 수:", int(round((1 - acc_split) * len(y_test))))  # 69마리 중


# ---------- 1-4 ----------
# 왜 · 분할 seed만 바꿔도 점수가 흔들린다는 것을 확인하기 위해
random_states_to_try = [51, 92, 14, 71, 60]                                  # 원본 Q5의 seed 5개
accs = []                                                                    # seed마다 test 정확도
print(f"{'random_state':>12s}  {'test 정확도':>10s}")                        # 표 머리
for rs in random_states_to_try:                                              # 시험지를 다시 뽑음
    # 새 분할
    Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, stratify=y, random_state=rs)
    m = DecisionTreeClassifier(random_state=42).fit(Xtr, ytr)                # 새로 학습
    accs.append(accuracy_score(yte, m.predict(Xte)))                         # 새 시험지로 채점
    print(f"{rs:>12d}  {accs[-1]:>10.4f}")                                   # 한 줄씩
print(f"범위 {min(accs):.4f} ~ {max(accs):.4f} · 평균 {np.mean(accs):.4f}")  # 흔들리는 폭

전체로 학습 + 전체로 채점: 1.0000
트리 잎 개수: 14
Train: 273마리 · Test: 69마리
전체 비율: {'Adelie': 0.442, 'Gentoo': 0.36, 'Chinstrap': 0.199}
Test 비율: {'Adelie': 0.435, 'Gentoo': 0.362, 'Chinstrap': 0.203}
[Q1] 전체로 학습+평가 (cheat) : 1.0000
[Q4] Train/Test 분리         : 0.9130
틀린 마리 수: 6
random_state    test 정확도
          51      0.9420
          92      0.9420
          14      0.9565
          71      0.9710
          60      0.9275
범위 0.9275 ~ 0.9710 · 평균 0.9478


**실행 흐름**

1. 2-1 교차검증 직접 풀기
2. 2-2 `cross_val_score`
3. 2-3 · 2-4 모델별 CV 평균 · 표준편차

In [18]:
from sklearn.linear_model import LogisticRegression  # 클래스별 확률 → 가장 높은 것
from sklearn.neighbors import KNeighborsClassifier   # 가까운 이웃 k개 다수결
from sklearn.model_selection import StratifiedKFold  # 품종 비율 유지 5등분
from sklearn.model_selection import cross_val_score  # 교차검증 한 줄


# ---------- 2-1 ----------

# 왜 · 같은 분할 · 같은 지표로 세 모델을 공정하게 비교하기 위해
models = {"DecisionTree": DecisionTreeClassifier(random_state=42),           # 질문을 던져 갈라 나감
          "LogisticRegression": LogisticRegression(max_iter=3000, random_state=42),  # 원본 Q6 설정
          "KNN": KNeighborsClassifier(n_neighbors=5)}                        # 이웃 5마리
print(f"{'모델':<20s}{'test 정확도':>10s}")                                  # 표 머리
for name, m in models.items():                                               # 같은 코드로 셋 다
    m.fit(X_train, y_train)                                                  # 같은 교과서
    print(f"{name:<20s}{accuracy_score(y_test, m.predict(X_test)):>10.4f}")  # 같은 시험지


# ---------- 2-2 ----------

# 왜 · 교차검증이 하는 일을 반복문으로 직접 풀어 이해하기 위해
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)                 # 섞어서 5등분
fold_scores = []  # fold마다 검증 정확도 (변수 이름 규약)
# (학습, 검증) 번호 쌍 5번
for k, (tr_idx, va_idx) in enumerate(kf.split(X_train, y_train), start=1):
    m = DecisionTreeClassifier(random_state=42)                                 # fold마다 새 모델
    m.fit(X_train[tr_idx], y_train[tr_idx])                                     # 4등분으로 공부
    # 1등분 모의고사
    fold_scores.append(accuracy_score(y_train[va_idx], m.predict(X_train[va_idx])))
    print(f"Fold {k}: {fold_scores[-1]:.4f} (학습 {len(tr_idx)} · 검증 {len(va_idx)})")  # fold마다
print(f"평균 {np.mean(fold_scores):.4f} · 표준편차 {np.std(fold_scores):.4f}")  # 요약


# ---------- 2-3 ----------

# 왜 · 같은 일을 cross_val_score 한 줄로 하고 결과를 비교하기 위해
tree = DecisionTreeClassifier(random_state=42)                    # 같은 설정
s5 = cross_val_score(tree, X_train, y_train, cv=5)                # 섞지 않는 5등분
print("cv=5 각 Fold:", s5.round(4), "평균", round(s5.mean(), 4))  # 원본 정답과 같은 형식
s_kf = cross_val_score(tree, X_train, y_train, cv=kf)             # Q7의 kf (섞은 5등분)
# 같은 분할
print("cv=kf 평균:", round(s_kf.mean(), 4), "· Q7 평균:", round(np.mean(fold_scores), 4))


# ---------- 2-4 ----------
# 왜 · 한 번의 분할 대신 교차검증 평균 · 표준편차로 모델을 비교하기 위해
print(f"{'모델':<20s}{'CV 평균':>10s}{'CV 표준편차':>12s}")  # 표 머리
cv_mean = {}                                                 # 정리표용
for name, m in models.items():                               # Q6 딕셔너리 그대로
    s = cross_val_score(m, X_train, y_train, cv=5)           # train 안에서만 5번
    cv_mean[name] = s.mean()                                 # 평균 저장
    print(f"{name:<20s}{s.mean():>10.4f}{s.std():>12.4f}")   # 평균 · 흔들림

모델                    test 정확도
DecisionTree            0.9130


LogisticRegression      0.9710
KNN                     0.7246
Fold 1: 0.9455 (학습 218 · 검증 55)
Fold 2: 0.9818 (학습 218 · 검증 55)
Fold 3: 0.9636 (학습 218 · 검증 55)
Fold 4: 0.9259 (학습 219 · 검증 54)
Fold 5: 0.9815 (학습 219 · 검증 54)
평균 0.9597 · 표준편차 0.0216
cv=5 각 Fold: [0.9636 0.9273 0.9818 0.963  1.    ] 평균 0.9671
cv=kf 평균: 0.9597 · Q7 평균: 0.9597
모델                       CV 평균     CV 표준편차
DecisionTree            0.9671      0.0241


LogisticRegression      0.9891      0.0145
KNN                     0.7801      0.0355


**실행 흐름**

1. 3-1 깊이별 교차검증
2. 3-2 GridSearchCV
3. 3-3 test 1회 채점

In [19]:
from sklearn.model_selection import GridSearchCV  # 후보 × 교차검증 자동화


# ---------- 3-1 ----------
# 왜 · 깊이를 바꿔 가며 교차검증 점수를 비교하는 과정을 직접 해 보기 위해
rows = []                                                                  # (깊이, 평균, 표준편차)
for d in [1, 2, 3, 4, None]:                                               # 원본 Q10 후보
    model = DecisionTreeClassifier(max_depth=d, random_state=42)           # 깊이만 바꿈
    s = cross_val_score(model, X_train, y_train, cv=5)                     # 5-Fold
    rows.append((str(d), s.mean(), s.std()))                               # None도 글자로
depth_df = pd.DataFrame(rows, columns=["max_depth", "cv_mean", "cv_std"])  # 표
print(depth_df.round(4).to_string(index=False))                            # 깊이별 점수


# ---------- 3-2 ----------

# 왜 · 직접 비교한 과정을 GridSearchCV로 자동화하기 위해
param_grid = {"max_depth": [1, 2, 3, 4, None]}                            # 사람이 정한 후보
grid = GridSearchCV(DecisionTreeClassifier(random_state=42), param_grid,  # 탐색할 모델 · 후보
                    cv=5, scoring="accuracy", return_train_score=True)    # 훈련 점수도 기록
grid.fit(X_train, y_train)  # 후보 5 × fold 5 = 25번 학습 + 최적값으로 train 전체 재학습
print("최적 파라미터:", grid.best_params_)                                # Q10에서 가장 높던 깊이
print(f"최적 CV 점수: {grid.best_score_:.4f}")                            # 고를 때 쓴 점수
cols = ["param_max_depth", "mean_train_score", "mean_test_score", "rank_test_score"]  # 볼 열
res = pd.DataFrame(grid.cv_results_)[cols]                                # 후보별 결과 표
print(res.round(4).to_string(index=False))                                # 훈련 vs 검증


# ---------- 3-3 ----------
# 왜 · 고를 때 쓰지 않은 test로 마지막에 한 번만 채점하기 위해
best_model = grid.best_estimator_  # 깊이 4로 train 전체를 다시 학습한 모델
final_acc = accuracy_score(y_test, best_model.predict(X_test))        # test 첫 사용
print(f"최적 파라미터        : {grid.best_params_}")                  # 고른 설정
print(f"최적 CV 점수 (train) : {grid.best_score_:.4f}")               # 고를 때 쓴 점수
print(f"최종 test 점수       : {final_acc:.4f}")                      # 보고할 점수
print(f"차이                 : {grid.best_score_ - final_acc:+.4f}")  # 낙관 정도

max_depth  cv_mean  cv_std
        1   0.7803  0.0104
        2   0.9306  0.0311
        3   0.9378  0.0376
        4   0.9708  0.0246
     None   0.9671  0.0241
최적 파라미터: {'max_depth': 4}
최적 CV 점수: 0.9708
param_max_depth  mean_train_score  mean_test_score  rank_test_score
              1            0.7903           0.7803                5
              2            0.9670           0.9306                4
              3            0.9835           0.9378                3
              4            0.9973           0.9708                1
           None            1.0000           0.9671                2
최적 파라미터        : {'max_depth': 4}
최적 CV 점수 (train) : 0.9708
최종 test 점수       : 0.9130
차이                 : +0.0578


**실행 흐름**

1. 방법별 점수 표 채우기
2. train 안에서 KNN의 k 고르기
3. test 1회

In [20]:
# ---------- 4. 종합 실습 ----------
# 1단계: 원본 '오늘 정리' 표 채우기
# 왜 · 방법마다 점수를 한 표에 모아 어떤 평가가 믿을 만한지 비교하기 위해
names = ["전체로 학습+평가 (cheat)", "Train/Test 분리",  # 원본 표의 방법 6개
         "5-Fold 평균 (DecisionTree)", "5-Fold 평균 (LogisticRegression)",
         "GridSearchCV 최적 CV 점수", "최종 홀드아웃 test 점수"]
vals = [acc_cheat, acc_split, cv_mean["DecisionTree"],   # 앞 장의 변수 그대로
        cv_mean["LogisticRegression"], grid.best_score_, final_acc]
summary = pd.DataFrame({"방법": names, "정확도": vals})  # 빈칸이 채워진 표
print(summary.round(4).to_string(index=False))           # 빈칸이 채워진 표


# 2 · 3단계: KNN의 k를 train 안에서 고르기 (test는 아직 봉인)
# 왜 · test를 고르는 데 쓰면 점수가 낙관적으로 나오므로 train 안에서만 고르기 위해
# 후보 6개
knn_grid = GridSearchCV(KNeighborsClassifier(), {"n_neighbors": [1, 3, 5, 7, 9, 15]}, cv=5)
knn_grid.fit(X_train, y_train)                                                   # 6 × 5 = 30번 학습
print("KNN 최적 k:", knn_grid.best_params_, f"· CV {knn_grid.best_score_:.4f}")  # 고른 k


# 4단계: test 1회
# 왜 · 선택이 끝난 뒤 test를 한 번만 써서 공정한 최종 점수를 얻기 위해
knn_final = accuracy_score(y_test, knn_grid.best_estimator_.predict(X_test))  # 처음이자 마지막
# 비교 (선택은 이미 끝남)
print(f"KNN 최종 test: {knn_final:.4f} · 트리 최종 test: {final_acc:.4f}")

                            방법    정확도
             전체로 학습+평가 (cheat) 1.0000
                 Train/Test 분리 0.9130
      5-Fold 평균 (DecisionTree) 0.9671
5-Fold 평균 (LogisticRegression) 0.9891
         GridSearchCV 최적 CV 점수 0.9708
               최종 홀드아웃 test 점수 0.9130


KNN 최적 k: {'n_neighbors': 1} · CV 0.8314
KNN 최종 test: 0.7971 · 트리 최종 test: 0.9130
